# Layer 4: Retrieval and RAG Safety Audit

This notebook audits the RAG layer without exposing secrets. It loads/builds the persisted ChromaDB index, runs retrieval-only checks, and provides an optional Claude synthesis cell that is disabled by default.


<!-- notebook-rationale -->
## Why this notebook is retrieval-first

For a public notebook, retrieval-only validation is safer and more reproducible than live LLM synthesis. The key engineering claim is that the system can retrieve relevant, ACN-cited incidents with useful metadata filters. Claude synthesis is useful for the demo, but it depends on an API key and should not be required for reviewers to understand the system.

The optional Claude cell is disabled by default and never prints secrets. This is intentional: safety systems and public repos should fail closed around credentials.


In [1]:
from pathlib import Path
import sys
import os

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "outputs" / "data"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CHROMA_DIR = DATA_DIR / "chromadb"

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
print("API key loaded: yes" if api_key else "API key loaded: no")


API key loaded: yes


In [2]:
from src.rag import DEMO_QUERIES, build_rag_index, rag_query, retrieve_incidents

## Load Layer 3 Corpus

Layer 4 indexes a prioritized subset of RED and ORANGE incidents from the full Layer 3 corpus.


<!-- notebook-rationale -->
## Why Layer 4 indexes RED and ORANGE

The RAG index is scoped to high-priority incidents rather than the full corpus. RED and ORANGE incidents are the analyst-relevant subset from Layer 1, and sorting by `precursor_score` prioritizes incidents with stronger human-factors signals.

This keeps retrieval focused. Indexing all GREEN routine incidents would increase noise and dilute the assistant's usefulness for early warning review.


In [3]:
layer3_path = DATA_DIR / "asrs_layer3.parquet"
assert layer3_path.exists(), "Run `uv run python run_layer3.py` first."
asrs = pd.read_parquet(layer3_path)
asrs["date"] = pd.to_datetime(asrs["date"], errors="coerce")
flagged = asrs[asrs["quadrant"].isin(["RED", "ORANGE"])]
print(f"Layer 3 shape: {asrs.shape}")
print(f"RED + ORANGE available: {len(flagged):,}")
assert len(flagged) == 6804


Layer 3 shape: (43829, 218)
RED + ORANGE available: 6,804


## Build or Load ChromaDB Index

`build_rag_index()` reuses a persisted index when metadata matches the expected version. If no index exists, this cell can rebuild it; that may take a few minutes on CPU.


<!-- notebook-rationale -->
## Why ChromaDB is acceptable for the prototype

ChromaDB gives local persistence with no server setup, which is appropriate for a case-study demo. The index is versioned in `src/rag.py` so stale indexes can be detected and rebuilt when important upstream assumptions change.

The production target would be Qdrant with hybrid dense/sparse retrieval. Aviation terms include acronyms and exact phrases where sparse matching matters, while analyst questions also benefit from semantic similarity.


In [4]:
collection, client, embedding_model = build_rag_index(asrs, max_incidents=3000, persist_dir=CHROMA_DIR, force_rebuild=False)
print(f"Indexed incidents: {collection.count():,}")
print(f"Vector store client: {type(client).__name__}")
assert collection.count() == 3000


2026-06-27 17:45:44 | INFO     | src.rag | Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-06-27 17:45:52 | INFO     | src.rag | Loaded existing index from C:\Users\Keshav.Gubbi\Documents\Code\asrs-early-warning\outputs\data\chromadb: 3,000 incidents
Indexed incidents: 3,000
Vector store client: Client


## Retrieval-Only Audit

This is the safest public demonstration: it exercises semantic retrieval and metadata filters without sending anything to an LLM.


<!-- notebook-rationale -->
## What retrieval-only proves

This cell proves that the vector index returns concrete incidents with ACN, date, quadrant, risk score, and similarity. That is the core evidence layer beneath any LLM answer.

A good RAG system should be useful even before generation. If the retrieved incidents are irrelevant, no prompt can make the answer trustworthy. This is why retrieval is audited separately from synthesis.


In [5]:
rows = retrieve_incidents(question="GPS navigation errors and unusual radar targets", collection=collection, embedding_model=embedding_model, n_results=5)
retrieval_table = pd.DataFrame([{"rank": row["rank"], "acn": row["metadata"].get("acn"), "date": row["metadata"].get("date"), "quadrant": row["metadata"].get("quadrant"), "risk": row["metadata"].get("precursor_score"), "similarity": round(row["similarity"], 3)} for row in rows])
display(retrieval_table)


,rank,acn,date,quadrant,risk,similarity
0,1,1725573,2020-02-01,ORANGE,0.275,0.670
1,2,1742053,2020-05-01,ORANGE,0.150,0.540
2,3,1912004,2022-06-01,ORANGE,0.125,0.514
3,4,1967829,2023-01-01,ORANGE,0.325,0.504
4,5,1949738,2022-11-01,ORANGE,0.350,0.489


## Metadata Filter Checks

The same retrieval layer can filter by quadrant or precursor score. This makes the RAG assistant useful for analyst workflows rather than generic semantic search.


<!-- notebook-rationale -->
## Why filters are part of the analyst workflow

Analysts rarely ask only broad semantic questions. They ask questions constrained by severity, novelty, time, or operational category. Metadata filtering lets the system answer those questions without forcing the LLM to sort through irrelevant context.

The RED-only and minimum-risk checks are examples of this workflow. They show that Layer 1 and Layer 3 signals survive into Layer 4 as actionable retrieval metadata.


In [6]:
red_rows = retrieve_incidents(question="What do RED quadrant incidents have in common?", collection=collection, embedding_model=embedding_model, n_results=5, filter_quadrant="RED")
fatigue_rows = retrieve_incidents(question="fatigue inadequate rest duty time", collection=collection, embedding_model=embedding_model, n_results=5, min_precursor_score=0.3)
red_table = pd.DataFrame([{"rank": row["rank"], "acn": row["metadata"].get("acn"), "quadrant": row["metadata"].get("quadrant"), "risk": row["metadata"].get("precursor_score")} for row in red_rows])
fatigue_table = pd.DataFrame([{"rank": row["rank"], "acn": row["metadata"].get("acn"), "quadrant": row["metadata"].get("quadrant"), "risk": row["metadata"].get("precursor_score")} for row in fatigue_rows])
display(red_table)
display(fatigue_table)
assert set(red_table["quadrant"]) == {"RED"}
assert (fatigue_table["risk"] >= 0.3).all()


,rank,acn,quadrant,risk
0,1,1915593,RED,0.075
1,2,1726432,RED,0.150
2,3,1909127,RED,0.150
3,4,1921731,RED,0.075
4,5,1743344,RED,0.125


,rank,acn,quadrant,risk
0,1,1906557,ORANGE,0.350
1,2,1689362,ORANGE,0.325
2,3,1704529,ORANGE,0.700
3,4,1993953,ORANGE,0.400
4,5,1693683,ORANGE,0.475


## Optional Claude Synthesis

This cell is disabled by default for public notebooks. Set `RUN_CLAUDE_DEMO = True` locally if you want to exercise answer synthesis. The code never prints the API key.


<!-- notebook-rationale -->
## Why synthesis is optional here

The Claude cell demonstrates the final analyst experience: a readable answer with ACN citations. It is disabled by default because public notebooks should not require secrets or make network calls during review.

When enabled locally, the system prompt constrains the model to retrieved reports and instructs it to state when evidence is insufficient. That is essential in a safety context: a no-evidence answer is better than a confident hallucination.


In [ ]:
RUN_CLAUDE_DEMO = False  # Set to True to run the Claude synthesis demo (requires ANTHROPIC_API_KEY)
if not RUN_CLAUDE_DEMO:
    print("Claude synthesis skipped. Set RUN_CLAUDE_DEMO = True to run locally.")
elif not api_key:
    print("Claude synthesis skipped because ANTHROPIC_API_KEY is not set.")
else:
    demo = DEMO_QUERIES[0]
    answer = rag_query(question=demo["question"], collection=collection, embedding_model=embedding_model, n_results=5, **demo.get("filter_kwargs", {}))
    print(answer)


## Pattern Analysis: GPS, Navigation Errors, and Unusual Radar Targets

### Overview

The retrieved incidents span 2020–2022, with **no incidents dated 2023** in this dataset. I must note this limitation explicitly: the evidence base does not support a 2023-specific analysis. The patterns below reflect what is available across the retrieved reports.

---

### Pattern 1: GPS Vulnerability to External Interference

Incidents [1] and [3] both document GPS/navigation system failures, though through different mechanisms:

- **[1]** describes **active radar jamming** from military assets lasting ~30 minutes, causing "multiple aircraft system failures" including corrupted Nav Display indications (anomalous colored radar "strikes" appeared on both pilots' displays). This was an ORANGE quadrant event with a moderate risk score (0.28).
- **[3]** documents a **spontaneous GPS position loss** in cruise, confirmed across both the primary indication and the FMS. Notably, the flight release included 

## Layer 4 to Layer 5 Handoff

Layer 4 is a demo-grade analyst assistant: it retrieves incidents and generates ACN-cited answers. Layer 5 describes the production version: multi-source investigation, sufficiency loops, self-critique, and human approval before publication.


<!-- notebook-rationale -->
## Why Layer 5 exists after RAG

Layer 4 answers individual analyst questions. Layer 5 describes the operational system: multi-source investigation, parallel retrieval workers, sufficiency checks, self-critique, and human approval before publication.

That distinction is important. A demo RAG assistant can support analysts, but an operational safety alerting system needs governance, audit logs, retry logic, and human-in-the-loop review.
